# Day 1 — Hands-on 6: Fine-tune a Pretrained Sentiment Model

**Goal:** Fine-tune DistilBERT on a small labelled sentiment dataset and compare evaluation before and after training.

**Recommended runtime:** Google Colab with GPU enabled (Runtime → Change runtime type → T4 GPU if available). CPU is supported but can be slow. This is a compact teaching experiment, not production training.

## Learning outcomes
Load a labelled dataset, tokenize examples, run a baseline, train with `Trainer`, evaluate the trained model, and identify where weights are updated.

In [1]:
!pip -q install "transformers>=4.40,<5" "datasets>=2.18" "accelerate>=0.28" torch scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.41.0 requires huggingface-hub<2.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

SEED = 42
MODEL_NAME = "distilbert/distilbert-base-uncased"
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

Device: CPU


## 1. Load a small subset of labelled data
We use the SST-2 sentence sentiment dataset. To keep the lab manageable, this exercise selects a small training and validation subset. Results will vary and should not be treated as benchmark-quality.

In [3]:
dataset = load_dataset("glue", "sst2")
train_ds = dataset["train"].shuffle(seed=SEED).select(range(500))
valid_ds = dataset["validation"].select(range(200))
print(train_ds[0])
print("Train rows:", len(train_ds), "Validation rows:", len(valid_ds))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

{'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1, 'idx': 32326}
Train rows: 500 Validation rows: 200


In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch["sentence"], truncation=True, max_length=128)

train_tok = train_ds.map(tokenize, batched=True)
valid_tok = valid_ds.map(tokenize, batched=True)
train_tok = train_tok.rename_column("label", "labels")
valid_tok = valid_tok.rename_column("label", "labels")
keep_cols = {"input_ids", "attention_mask", "labels"}
train_tok = train_tok.remove_columns([c for c in train_tok.column_names if c not in keep_cols])
valid_tok = valid_tok.remove_columns([c for c in valid_tok.column_names if c not in keep_cols])
collator = DataCollatorWithPadding(tokenizer=tokenizer)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

## 2. Evaluate the pretrained model before fine-tuning

In [5]:
baseline_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1}

baseline_args = TrainingArguments(
    output_dir="/content/baseline_eval",
    per_device_eval_batch_size=16,
    report_to="none",
    seed=SEED
)
baseline_trainer = Trainer(model=baseline_model, args=baseline_args,
    eval_dataset=valid_tok, tokenizer=tokenizer, data_collator=collator, compute_metrics=compute_metrics)
baseline = baseline_trainer.evaluate()
print({k: round(v, 4) for k, v in baseline.items() if k.startswith("eval_")})

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_2591/1041963993.py:14: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  baseline_trainer = Trainer(model=baseline_model, args=baseline_args,
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': 0.6954, 'eval_model_preparation_time': 0.0096, 'eval_accuracy': 0.485, 'eval_precision': 0.1667, 'eval_recall': 0.0101, 'eval_f1': 0.019, 'eval_runtime': 47.2933, 'eval_samples_per_second': 4.229, 'eval_steps_per_second': 0.275}


## 3. Fine-tune: this is where weight updates happen

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
training_args = TrainingArguments(
    output_dir="/content/distilbert-sst2-finetuned",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_steps=20,
    report_to="none",
    seed=SEED,
    fp16=torch.cuda.is_available()
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_tok,
    eval_dataset=valid_tok, tokenizer=tokenizer, data_collator=collator, compute_metrics=compute_metrics)
train_result = trainer.train()
print("Training completed. Global steps:", trainer.state.global_step)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert/distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_2591/2132957175.py:16: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=training_args, train_dataset=train_tok,
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.638900,0.637448,0.560000,0.530055,0.979798,0.687943


Training completed. Global steps: 63


## 4. Evaluate after fine-tuning

In [8]:
after = trainer.evaluate()
print("BEFORE:", {k: round(v, 4) for k, v in baseline.items() if k.startswith("eval_")})
print("AFTER: ", {k: round(v, 4) for k, v in after.items() if k.startswith("eval_")})

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


BEFORE: {'eval_loss': 0.6954, 'eval_model_preparation_time': 0.0096, 'eval_accuracy': 0.485, 'eval_precision': 0.1667, 'eval_recall': 0.0101, 'eval_f1': 0.019, 'eval_runtime': 47.2933, 'eval_samples_per_second': 4.229, 'eval_steps_per_second': 0.275}
AFTER:  {'eval_loss': 0.6374, 'eval_accuracy': 0.56, 'eval_precision': 0.5301, 'eval_recall': 0.9798, 'eval_f1': 0.6879, 'eval_runtime': 14.8132, 'eval_samples_per_second': 13.501, 'eval_steps_per_second': 0.878}


In [9]:
# Try your own text using the fine-tuned model
from transformers import pipeline
fine_tuned_sentiment = pipeline("sentiment-analysis", model=trainer.model, tokenizer=tokenizer,
                                device=0 if torch.cuda.is_available() else -1)
for text in ["The service was excellent.", "This was a terrible experience.", "It was fine, nothing special."]:
    print(text, "->", fine_tuned_sentiment(text)[0])

Device set to use cpu


The service was excellent. -> {'label': 'LABEL_1', 'score': 0.6486151218414307}
This was a terrible experience. -> {'label': 'LABEL_1', 'score': 0.5484825968742371}
It was fine, nothing special. -> {'label': 'LABEL_1', 'score': 0.6049222350120544}


## 5. Save the fine-tuned model (optional)
This saves model files in the current runtime. To keep them after a Colab session ends, download or copy the folder to persistent storage.

In [10]:
SAVE_DIR = "/content/my-finetuned-sentiment-model"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)

Saved to: /content/my-finetuned-sentiment-model


## Student assignment and reflection
1. Run the notebook and record baseline and post-training accuracy/F1.
2. Explain which cell performs training and why it updates weights.
3. Did metrics improve? If not, give two plausible reasons (small sample, one epoch, randomness, distribution differences).
4. Change `num_train_epochs` from 1 to 2 and compare. Do not assume more epochs always means better generalization.
5. Explain the difference between training data and validation data.

**Important:** This notebook fine-tunes a pretrained encoder for classification; it is not full pretraining of a large generative LLM. Avoid describing it as training an LLM from scratch.

## Troubleshooting
- If `eval_strategy` is unrecognized, upgrade Transformers and restart the runtime.
- If memory errors occur, lower batch sizes or use fewer examples.
- If the dataset download fails, check internet access.
- Metrics on this small subset are noisy; report the subset size and settings.